We will now analyze the actual data

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from anndata.io import read_elem


peek at the structure of the .h5ad file (obs, var, layers) without loading the 1.19M x 33.5k count matrix, which is too big for memory.

In [ ]:
with h5py.File(r"..\data\Full_cohort.h5ad", "r") as f:
    print('X type:', type(f['X']))  # Print the type of the 'X' dataset
    print(list(f.keys()))  # List the top-level groups in the HDF5 file
    print('obs keys:', list(f['obs'].keys()))  # List the keys in the 'obs' group
    print('var keys:', list(f['var'].keys()))  # List the keys in the 'var' group
    print('uns keys:', list(f['uns'].keys()))  # List the keys in the 'uns' group
    print('layers keys:', list(f['layers'].keys()))  # List the keys in the 'layers' group
    
    for layer in f['layers']:
        print('\n',layer)
        print(f"type of {layer}:", type(f['layers'][layer]))  # Print the type of each layer
        print(f"keys in {layer}:", list(f['layers'][layer].keys()))  # List the keys in each layer
        print(f"attributes of {layer}:", list(f['layers'][layer].attrs))  # List the attributes of each layer

In [ ]:
with h5py.File(r"..\data\Full_cohort.h5ad", "r") as f:
    print(f"Shape of X: {f['X'].attrs['shape']}")  # Print the shape of the 'X' dataset
    print(f"Shape of layer counts: {f['layers']['counts'].attrs['shape']}")  # Print the shape of the 'counts' layer
    print(f"Shape of layer log1p_cp10k: {f['layers']['log1p_cp10k'].attrs['shape']}")  # Print the shape of the 'logcounts' layer

In [ ]:
with h5py.File(r"..\data\Full_cohort.h5ad", "r") as f:
    obs = read_elem(f['obs'])
    var = read_elem(f['var'])
    print(type(obs))
    print(obs.shape)
    print(type(var))
    print(var.shape)

In [ ]:
obs.head()

see how many cells each broad cell type has, to know which types are common or rare.

In [ ]:
obs['category'].value_counts()

Check the spread of genes detected per cell; very low values flag empty droplets or low-quality cells.

In [ ]:
obs['n_genes_by_counts'].hist(bins=50)

check column types and missing values in obs before using any column.

In [ ]:
obs.info()

In [ ]:
obs.describe()

In [ ]:
var.head()

confirm every feature is a gene, with no other feature types mixed in.

In [ ]:
var['feature_types'].value_counts()

count the genes flagged as mitochondrial; they define the mito % QC metric.

In [ ]:
var['gene_group__mito_transcript'].value_counts()

count the ribosomal-protein genes, which we plan to drop before computing gene correlations.

In [ ]:
var['gene_group__ribo_protein'].value_counts()

check the ribosomal-RNA flag; if it is False for every gene it carries no information here.

In [ ]:
var['gene_group__ribo_rna'].value_counts()

In [ ]:
var['pct_dropout_by_counts'].describe()

Join metadata to obs

before joining, check that the sample IDs in the metadata match those in the expression data, so the join does not silently lose cells.

In [ ]:
# check if sample ids in metadata match the sample ids in the expression matrix
mapped_df = pd.read_csv(r"..\data\mapped_data.tsv", sep="\t", index_col=0)
sample_ids_in_metadata = set(mapped_df.index)
sample_ids_in_expression_matrix =  set(obs['sanger_sample_id'])

# Find the intersection of sample IDs
common_sample_ids = sample_ids_in_metadata & sample_ids_in_expression_matrix

# Find the difference of sample IDs
only_in_metadata = sample_ids_in_metadata - sample_ids_in_expression_matrix
only_in_expression_matrix = sample_ids_in_expression_matrix - sample_ids_in_metadata

In [ ]:
print(f"Number of sample IDs in metadata: {len(sample_ids_in_metadata)}")
print(f"Number of sample IDs in expression matrix: {len(sample_ids_in_expression_matrix)}")
print(f"Number of common sample IDs: {len(common_sample_ids)}")
print(f"Number of sample IDs only in metadata: {len(only_in_metadata)}")
print(f"Number of sample IDs only in expression matrix: {len(only_in_expression_matrix)}")

In [ ]:
sample_ids_common = obs['sanger_sample_id'].isin(mapped_df.index).sum()
sample_ids_not_common = (~obs['sanger_sample_id'].isin(mapped_df.index)).sum()
print(sample_ids_common, sample_ids_not_common)

In [ ]:
mapped_df.head()

In [ ]:
obs.head()

sanity check that every cell ID appears once, so each row of obs means exactly one cell.

In [ ]:
# check if the index of obs is unique
obs.index.nunique() == obs.shape[0]

Join the 2 tables on sanger sample id

In [ ]:
obs = obs.join(mapped_df, on='sanger_sample_id', how='left')

confirm the join did not change the number of cells.

In [ ]:
obs.shape

In [ ]:
obs.describe()

Confirm the column types after the join and see how many values are non-null in each column.

In [ ]:
obs.info()

Count missing values per column to find which cells lack metadata.

In [ ]:
obs.isna().sum()

Isolate the cells with no donor information so we can see what they are.

In [ ]:
# rows with no induvidual information
no_individual_info = obs[obs['individual_id'].isnull()]
print(f"Number of rows with no individual information: {no_individual_info.shape[0]}")
no_individual_info.head()

Find which sample these cells come from.

In [ ]:
no_individual_info['sanger_sample_id'].value_counts()

In [ ]:
print('unique samples:', obs['sanger_sample_id'].nunique())
print('unique individuals:', obs['individual_id'].nunique())

Let us drop the rows without donor data - 5850. remove the cells with no donor information, because we cannot assign them to a health group.

In [ ]:
obs = obs.dropna(subset=['individual_id'])
obs['individual_id'].isnull().sum()

In [ ]:
obs['health_category'].value_counts()

See the cell-type makeup within each health group

In [ ]:
obs.groupby('category')['health_category'].value_counts()

Sanity check that every donor belongs to exactly one health category.

In [ ]:
obs.groupby('individual_id')['health_category'].nunique().sum() == obs['individual_id'].nunique()

Check cell statistics

In [ ]:
obs.groupby('health_category')['individual_id'].nunique()

In [ ]:
obs['individual_id'].value_counts().aggregate(['mean', 'median', 'min', 'max'])

In [ ]:
# cells per donor
cells_per_donor = obs.groupby('individual_id').size()
cells_per_donor.describe()

Look at the shape of the cells-per-donor distribution on a log scale, with the median marked.

In [ ]:
# histogram plot of cells per donor
plt.figure(figsize=(10, 6))
plt.hist(cells_per_donor, bins=np.logspace(np.log10(cells_per_donor.min()),np.log10(cells_per_donor.max()), 30))
plt.axvline(cells_per_donor.median(), color='red', ls='--', label='median')
plt.legend()
plt.xlabel('Cells per Donor')
plt.ylabel('Frequency')
plt.title('Distribution of Cell Counts per Donor')
plt.show()

Compare linear and log scales to show why the log scale is easier to read for skewed counts.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].hist(cells_per_donor, bins=30)
ax[0].set_title('Linear')
ax[1].hist(np.log10(cells_per_donor), bins=30)
ax[1].set_title('Logarithmic')
ax[0].set_xlabel('Cells per Donor')
ax[1].set_xlabel('Log10(Cells per Donor)')
plt.show()

cell-type table of cell counts (plus a total column)

In [ ]:
cells_per_donor_celltypes = obs.groupby(['individual_id', 'category', 'health_category']).size().unstack('category', fill_value=0)
cells_per_donor_celltypes['total'] = cells_per_donor_celltypes.sum(axis=1)
cells_per_donor_celltypes.head()

In [ ]:
# how many donors per group, and cells per donor
cells_per_donor_celltypes.groupby('health_category')['total'].describe()

Compare cells per donor across cell types on a log scale to see which types are abundant and which are sparse.  
Total number of cells is included

In [ ]:
# boxplot of cells per donor by health category
sns.boxplot(data=cells_per_donor_celltypes, orient='h', palette='Set2')
plt.xlabel('Cells per donor')
plt.xscale('log')
plt.show()

Convert counts to fractions of each donor's cells, so donors with different total cell numbers become comparable.

In [ ]:
fractions = cells_per_donor_celltypes.div(cells_per_donor_celltypes['total'], axis=0).reset_index()
fractions.drop(columns='total', inplace=True)
sns.boxplot(data=fractions, orient='h', palette='Set2')
plt.xlabel('Fraction of Cells per Donor')
plt.ylabel('Cell Type')

Why: check whether total cells per donor differs by health group, since that would distort raw-count comparisons.

In [ ]:
# total cells per donor by health category
sns.boxplot(data=cells_per_donor_celltypes, x='health_category', y='total', hue='health_category', palette='Set2')

Reshape the fractions table from wide to long format, which seaborn needs for grouped plots.

In [ ]:
fraction_df = fractions.melt(id_vars=['health_category', 'individual_id'], var_name='category', value_name='fraction')
fraction_df['category'].value_counts()

Compare each cell type's fraction between health groups - does the cell mix change with disease?

In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(data=fraction_df, x='category', y='fraction', hue='health_category', palette='Set2')

conf_score is stored as text (category), so convert it to float before filtering or plotting it.

In [ ]:
obs['conf_score'] = obs['conf_score'].astype(float)
obs['conf_score'].describe()

count cells whose cell-type label has confidence below 0.5, to decide whether low-confidence labels need removing.

In [ ]:
(obs['conf_score']<0.5).sum()

In [ ]:
cells_per_donor_celltypes.head()

In [ ]:
# donors per group per cell type
type_counts = cells_per_donor_celltypes.drop(columns='total')

In [ ]:
donor_per_type = (type_counts >= 20).groupby('health_category').sum()
print(donor_per_type)

donor_per_type.T.plot(kind='bar', figsize=(10, 5), colormap='Set2', rot=45, stacked=True)
plt.ylabel(f'Donors with at least {20} cells')
plt.show()


In [ ]:
min_cells = [20, 50, 75, 100]
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharey=True)

for ax, n in zip(axes.flat, min_cells):
    frac = (type_counts >= n).groupby('health_category').mean()
    frac.T.plot(kind='bar', ax=ax, colormap='Set2', rot=45, legend=(ax is axes[0, 0]), )
    ax.set_title(f'At least {n} cells')
    ax.set_ylim(0, 1)
    ax.set_xlabel('')

axes[0, 0].set_ylabel('Fraction of donors')
axes[1, 0].set_ylabel('Fraction of donors')
fig.tight_layout()
plt.show()


We will choose Enterocyte and T cell as the priority to analyze further. This is because of their abanduncy and balanced distribution on initial look among different health groups

Enterocyte analysis

In [ ]:
ent_obs = obs[obs['category']=='Enterocyte'].copy()
ent_obs['label'] = ent_obs['label'].cat.remove_unused_categories()

In [ ]:
print(ent_obs.shape, obs.shape[0] - ent_obs.shape[0])

In [ ]:
ent_obs['label'].value_counts()

In [ ]:
ent_obs.groupby(['label', 'health_category']).size()

In [ ]:
ent_counts = (ent_obs.groupby(['individual_id', 'health_category', 'label'], observed=True)
                     .size().unstack('label', fill_value=0))

ent_counts = ent_counts[ent_counts.sum(axis=1) >= 50]   # drop donors with too few enterocytes

ent_frac = ent_counts.div(ent_counts.sum(axis=1), axis=0).reset_index()
ent_long = ent_frac.melt(id_vars=['individual_id', 'health_category'],
                         var_name='label', value_name='fraction')

plt.figure(figsize=(10, 8))
sns.boxplot(data=ent_long, y='label', x='fraction', hue='health_category',
            hue_order=['Healthy', 'CD uninflamed', 'CD inflamed'], palette='Set2', fliersize=2)
plt.xlabel("Fraction of the donor's enterocytes")
plt.show()


Enterocytes precursor crypt KRT20+ has more CD inflamed cells than healthy or uninflamed.  
Enterocytes BEST4+ has too less cells

In [ ]:
t_obs = obs[obs['category']=='T cell'].copy()
t_obs['label'] = t_obs['label'].cat.remove_unused_categories()
t_obs.shape

In [ ]:
t_obs['label'].value_counts()

In [ ]:
t_obs.groupby(['label', 'health_category']).size()

In [ ]:
t_counts = (t_obs.groupby(['individual_id', 'health_category', 'label'], observed=True)
                     .size().unstack('label', fill_value=0))

t_counts = t_counts[t_counts.sum(axis=1) >= 50]   # drop donors with too few t cells

t_frac = t_counts.div(t_counts.sum(axis=1), axis=0).reset_index()
t_long = t_frac.melt(id_vars=['individual_id', 'health_category'],
                         var_name='label', value_name='fraction')

plt.figure(figsize=(10, 8))
sns.boxplot(data=t_long, y='label', x='fraction', hue='health_category',
            hue_order=['Healthy', 'CD uninflamed', 'CD inflamed'], palette='Set2', fliersize=2)
plt.xlabel("Fraction of the donor's t cells")
plt.show()


In [ ]:
ent_mean = ent_frac.drop(columns='individual_id').groupby('health_category').mean().T*100
ent_mean.round(2).sort_values('CD inflamed', ascending=False)

In [ ]:
t_mean = t_frac.drop(columns='individual_id').groupby('health_category').mean().T*100
t_mean.round(2).sort_values('CD inflamed', ascending=False)

Enterocytes precursor crypt KRT20+	varies from 21.07% for CD inflamed to 1.68% for healthy donors.  
T cells CD4+ Treg is the biggest difference with variation from 8.23% in CD inflamed to 2.07% in healthy donors

Since T cells fractions for donors is more stable across different health catgeories compared to Enterocytes, we will be using Tcells for coexpression